# Water use processing

Create a single netcdf with all variables

In [ ]:
# basic
from pathlib import Path
import pandas as pd
import yaml
import os

# spatial
import geopandas as gpd
import xarray as xr

path_data_raw = Path("/home/rooda/Datasets")
wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd)

# import config 
with open(wd / "config.yml") as f:
    cfg = yaml.safe_load(f)

from src.utils import reshape_dataset, load_and_convert_csv_to_geocube

## Huang et al.

Global monthly gridded (0.5°) water use dataset (1971–2010) covering multiple sectors with withdrawals (withd_*) and consumptive use (cons_*):

- **Domestic**: withd_dom.nc, cons_dom.nc  
- **Electricity**: withd_elec.nc, cons_elec.nc  
- **Irrigation**: withd_irr.nc, cons_irr.nc (four GHM variants)
- **Livestock**: withd_liv.nc, cons_liv.nc  
- **Manufacturing**: withd_mfg.nc, cons_mfg.nc  
- **Mining**: withd_min.nc, cons_min.nc

In [ ]:
list_netcdf = sorted((path_data_raw / "WATER_USE/Huang_HESS").rglob("*.nc"))

datasets = []
for file in list_netcdf:
    with xr.open_dataset(file, decode_times=False, decode_coords=False) as ds:
        da = reshape_dataset(ds).sel(
            lon=slice(cfg["area"]["min_lon"], cfg["area"]["max_lon"]),
            lat=slice(cfg["area"]["min_lat"], cfg["area"]["max_lat"]),
        )
        datasets.append(da)

# keep only consensus where duplicate variable names exist
merged_ds = xr.merge(datasets, compat="override")

withd_vars = [v for v in merged_ds.data_vars if v.startswith("withd")]
cons_vars = [v for v in merged_ds.data_vars if v.startswith("cons")]

merged_ds["withd_total"] = sum(merged_ds[v] for v in withd_vars)
merged_ds["cons_total"] = sum(merged_ds[v] for v in cons_vars)

merged_ds = merged_ds.astype("float32")

# the units are in mm/month, as per the original data source (Huang et al., HESS)
for var in merged_ds.data_vars:
    merged_ds[var].attrs["units"] = "mm/month"
merged_ds.attrs["units"] = "mm/month"

encoding = {var: {"compression": "zstd"} for var in merged_ds.data_vars}
merged_ds.to_netcdf(path_data_raw / "WATER_USE/Huang_HESS/Water_use_Huang.nc", encoding=encoding)

## CR2WU

The CR2WU dataset includes water use estimates for consumptive uses across all sectors and non-consumptive uses for sectors with major return flows. Two distinct methodologies are employed:

1. **LULUCF sectors** (Land Use, Land-Use Change, and Forestry): Irrigated agriculture and rainfed agriculture use an evapotranspiration (ET) scheme for water use estimation.

2. **Other sectors**: Drinking water, energy, mining, manufacturing industry, and livestock are estimated using methodologies from previous studies commissioned by DGA, with corresponding subsectors detailed in the literature.

In [ ]:
# commune shapefile
shape = gpd.read_file(path_data_raw / "WATER_USE/CR2WU/comunas.gpkg")
shape = shape.set_index("cod_comuna").sort_index()
shape.index = shape.index.astype(str)
shape = shape[["Region", "Comuna", "Provincia", "geometry"]]
shape = shape.to_crs("EPSG:32719")
shape["area_m2"] = shape.geometry.area 
shape = shape.to_crs("EPSG:4326")
shape = shape.drop(["5201", "5104", "0"]) # remove Isla de Pascua

In [ ]:
years = list(range(1960, 2021))
res = (-0.05, 0.05)

### Other sectors (NLU)

In [ ]:
sectors = ["DrinkingWater", "UnbilledDrinkingWater", "Mining", "Energy", "Industry", "Livestock"]
files = sorted((path_data_raw / "WATER_USE/CR2WU/CR2WU_NLU").glob("*.csv"))

final_dfs = []

for sector_idx, sector_name in enumerate(sectors, 1):
    print(f"Processing sector {sector_idx}: {sector_name}")
    
    # Filter files for this sector
    filtered_files = [f for f in files if f.name.split('_')[2].startswith(str(sector_idx))]
    
    # Load and sum all CSVs for this sector
    dfs = [pd.read_csv(f, index_col=0) for f in filtered_files]
    sector_df = dfs[0].add(*dfs[1:], fill_value=0) if len(dfs) > 1 else dfs[0]
    
    # Convert to geocube and append
    geocube = load_and_convert_csv_to_geocube(sector_df, sector_name, years=years, shape=shape, res=res)
    final_dfs.append(geocube)

# Merge all sectors and add total
final_df = xr.merge(final_dfs)
final_df["Total_NLU"] = sum(final_df[sector] for sector in sectors)

# Save with compression
encoding = {var: {"compression": "zstd"} for var in final_df.data_vars}
final_df.to_netcdf(path_data_raw / "WATER_USE/CR2WU/CR2WU_NLU_1960_2020.nc", encoding=encoding)


### LULUCF sectors

In [ ]:
df_irr_con = load_and_convert_csv_to_geocube(
    path_data_raw / "WATER_USE/CR2WU/CR2WU_LULUCF/CR2WU_LU_v1.1_Irrigated_Consumptive_1960_2020_commune.csv", 
    "irr_con",
    years=years, 
    shape=shape, 
    res=res
).rename("Irrigated_Consumptive")

df_rain_con = load_and_convert_csv_to_geocube(
    path_data_raw / "WATER_USE/CR2WU/CR2WU_LULUCF/CR2WU_LU_v1.1_Rainfed_Consumptive_1960_2020_commune.csv", 
    "rain_con",
    years=years, 
    shape=shape, 
    res=res
).rename("Rainfed_Consumptive")

df_total_con = load_and_convert_csv_to_geocube(
    path_data_raw / "WATER_USE/CR2WU/CR2WU_LULUCF/CR2WU_LU_v1.1_Total_Consumptive_1960_2020_commune.csv", 
    "total_con",
    years=years, 
    shape=shape, 
    res=res
).rename("Total_Consumptive")

final_df = xr.merge([df_irr_con, df_rain_con, df_total_con])

encoding = {var: {"compression": "zstd"} for var in final_df.data_vars}
final_df.to_netcdf(path_data_raw / "WATER_USE/CR2WU/CR2WU_LULUCF_1960_2020.nc", encoding=encoding)